# XTTS-v2 Fine-tuning: Train Your Own Voice

## Why Fine-tune?
Voice cloning (notebook 03) works well, but fine-tuning gives you:
- **Better voice accuracy** — closer match to target speaker
- **Consistency** — same quality across all generations
- **Speed** — no need to process reference audio each time

### Fine-tuning vs. Voice Cloning
```
Voice Cloning (zero-shot):           Fine-tuning:
┌──────────┐                         ┌──────────────────┐
│ 6s audio │──▶ Speaker Encoder      │ 10-30 min audio  │
└──────────┘    (frozen model)       │ + transcriptions │
     │                                └────────┬─────────┘
     ▼                                         │
  Quick but                              Train GPT-2 decoder
  approximate                            for ~5 epochs
                                               │
                                               ▼
                                         Precise &
                                         consistent
```

**Data needed**: 10-30 min of clean speech + transcriptions  
**Time**: ~2-4 hours on Kaggle T4 GPU  
**Dataset to attach**: `luizfelipebjcosta/libri-tts-train-clean-100`

---
**Kaggle Setup**: GPU T4 x2 | 12h session limit

## Step 1: Install & Setup

In [ ]:
!pip install -q TTS==0.22.0 pydub librosa matplotlib pandas
!pip install -q --force-reinstall 'protobuf>=5.26.1'

In [ ]:
import torch
import torchaudio
import numpy as np
import pandas as pd
import os
import time
import librosa
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"GPU: {gpu} ({vram / 1e9:.1f} GB)")

## Step 2: Prepare Training Data

XTTS fine-tuning needs:
- **WAV files**: 22050 Hz, mono, 2-12 seconds each
- **Metadata CSV**: `audio_path|text|speaker_name`

We'll use **LibriTTS** — a clean speech dataset from audiobooks.  
**Attach** `luizfelipebjcosta/libri-tts-train-clean-100` in Kaggle.

In [ ]:
# ──────────────────────────────────────────────────────────
# CONFIGURE THIS
# ──────────────────────────────────────────────────────────
LIBRI_PATH = Path("/kaggle/input/libri-tts-train-clean-100")
SPEAKER_ID = "19"   # Pick a speaker. Try: 19, 26, 39, 40
MAX_FILES = 80       # Use 80-200 clips for fine-tuning
OUTPUT_DIR = Path("/kaggle/working/xtts_ft")
# ──────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find all audio files for this speaker
search_dirs = [
    LIBRI_PATH / "LibriTTS" / "train-clean-100" / SPEAKER_ID,
    LIBRI_PATH / "train-clean-100" / SPEAKER_ID,
    LIBRI_PATH / SPEAKER_ID,
]

wav_files = []
for d in search_dirs:
    if d.exists():
        wav_files = sorted(d.rglob("*.wav"))[:MAX_FILES]
        print(f"Found {len(wav_files)} files in {d}")
        break

if not wav_files:
    print("Dataset not found! Make sure to attach it in Kaggle.")
    print(f"Searched: {[str(d) for d in search_dirs]}")

In [ ]:
# Explore the data: listen to a few samples
if wav_files:
    print(f"Speaker {SPEAKER_ID} — sample clips:")
    for wav in wav_files[:3]:
        # Read transcript
        txt = wav.with_suffix(".normalized.txt")
        if not txt.exists():
            txt = wav.with_suffix(".txt")
        transcript = txt.read_text().strip() if txt.exists() else "(no transcript)"
        
        # Audio info
        info = torchaudio.info(str(wav))
        dur = info.num_frames / info.sample_rate
        
        print(f"\n📁 {wav.name} ({dur:.1f}s, {info.sample_rate}Hz)")
        print(f"   \"{transcript[:80]}\"")
        display(Audio(str(wav)))

In [ ]:
# Analyze the dataset
if wav_files:
    durations = []
    for wav in wav_files:
        info = torchaudio.info(str(wav))
        durations.append(info.num_frames / info.sample_rate)
    
    total_min = sum(durations) / 60
    
    print(f"Dataset stats for speaker {SPEAKER_ID}:")
    print(f"  Clips: {len(durations)}")
    print(f"  Total: {total_min:.1f} minutes")
    print(f"  Mean:  {np.mean(durations):.1f}s")
    print(f"  Min:   {min(durations):.1f}s")
    print(f"  Max:   {max(durations):.1f}s")
    
    plt.figure(figsize=(8, 3))
    plt.hist(durations, bins=20, edgecolor='black', alpha=0.7)
    plt.xlabel('Duration (seconds)')
    plt.ylabel('Count')
    plt.title(f'Clip Durations — Speaker {SPEAKER_ID} ({total_min:.1f} min total)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Create metadata CSV for training
# Format: audio_path|text|speaker_name

records = []
skipped = 0

for wav in wav_files:
    # Get transcript
    txt = wav.with_suffix(".normalized.txt")
    if not txt.exists():
        txt = wav.with_suffix(".txt")
    if not txt.exists():
        skipped += 1
        continue
    
    text = txt.read_text().strip()
    
    # Filter: skip clips shorter than 2s or longer than 12s
    info = torchaudio.info(str(wav))
    dur = info.num_frames / info.sample_rate
    if dur < 2.0 or dur > 12.0:
        skipped += 1
        continue
    
    records.append({
        "audio_file": str(wav),
        "text": text,
        "speaker_name": f"speaker_{SPEAKER_ID}"
    })

df = pd.DataFrame(records)
meta_path = OUTPUT_DIR / "metadata_train.csv"
df.to_csv(meta_path, sep="|", index=False, header=False)

print(f"Training metadata: {len(df)} clips ({skipped} skipped)")
print(f"Saved to: {meta_path}")
df.head()

## Step 3: Fine-tune XTTS-v2

### What gets trained?
Only the **GPT-2 decoder** is fine-tuned. The speaker encoder and vocoder stay frozen.  
This is called **adapter fine-tuning** — fast and memory-efficient.

```
Speaker Encoder  ──▶ [FROZEN]     ──▶ Speaker Embedding
                                            │
Text Encoder     ──▶ [FROZEN]     ──▶       │
                                            ▼
                     [GPT-2 Decoder]  ◀── TRAINED ✏️
                            │
                            ▼
HiFi-GAN Vocoder ──▶ [FROZEN]     ──▶ 🔊 Audio
```

In [ ]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from TTS.config.shared_configs import BaseDatasetConfig

# Training hyperparameters
# ──────────────────────────────────────────────────────────
NUM_EPOCHS = 5          # 5-10 for quick results, 20+ for best
BATCH_SIZE = 2          # 2-4 depending on VRAM
LEARNING_RATE = 5e-6    # Low LR for fine-tuning
EVAL_SPLIT = 0.1        # 10% for validation
# ──────────────────────────────────────────────────────────

print(f"Training plan:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Training clips: {len(df)}")
print(f"  Steps/epoch: ~{len(df) // BATCH_SIZE}")
print(f"  Total steps: ~{(len(df) // BATCH_SIZE) * NUM_EPOCHS}")

In [ ]:
# Run fine-tuning using the Coqui TTS Trainer
# This is the simplest approach

import json

# Create a training config
train_config = {
    "output_path": str(OUTPUT_DIR / "training"),
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "metadata_path": str(meta_path),
    "language": "en",
}

config_path = OUTPUT_DIR / "train_config.json"
with open(config_path, "w") as f:
    json.dump(train_config, f, indent=2)

print(f"Config saved to {config_path}")
print("\nTo start training, uncomment and run the cell below.")
print("Estimated time: 2-4 hours on T4 GPU")

In [ ]:
# ⚠️ UNCOMMENT TO START TRAINING (takes 2-4 hours)

# from TTS.api import TTS
# from TTS.utils.manage import ModelManager
# 
# # Download base model first
# tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
# model_path = tts.model_path
# del tts  # Free memory
# 
# # Start fine-tuning via CLI
# !python -m TTS.bin.train_tts \
#     --config_path {model_path}/config.json \
#     --coqpit.output_path {OUTPUT_DIR}/training \
#     --coqpit.datasets.0.formatter ljspeech \
#     --coqpit.datasets.0.meta_file_train metadata_train.csv \
#     --coqpit.datasets.0.path {OUTPUT_DIR} \
#     --coqpit.datasets.0.language en \
#     --coqpit.run_eval false \
#     --coqpit.epochs {NUM_EPOCHS} \
#     --coqpit.batch_size {BATCH_SIZE} \
#     --coqpit.lr {LEARNING_RATE}

## Step 4: Test the Fine-tuned Model

In [ ]:
# After training completes, test the model
from TTS.api import TTS

# Load fine-tuned model (update path after training)
FINETUNED_PATH = OUTPUT_DIR / "training"  # contains best_model.pth + config.json

best_model = FINETUNED_PATH / "best_model.pth"
ft_config = FINETUNED_PATH / "config.json"

if best_model.exists():
    print("Loading fine-tuned model...")
    tts = TTS(
        model_path=str(best_model),
        config_path=str(ft_config),
    ).to(device)
    print("Fine-tuned model loaded!")
else:
    print("No fine-tuned model found. Using base model for demo.")
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

In [ ]:
# A/B test: compare base model vs fine-tuned
test_texts = [
    "The sun was setting behind the mountains, painting the sky in shades of orange and purple.",
    "Machine learning has revolutionized how we approach complex problems in science and engineering.",
    "Would you like a cup of coffee? I just made a fresh pot this morning.",
]

# Use a reference clip from the training data
if wav_files:
    ref_wav = str(wav_files[0])
    print(f"Reference audio: {Path(ref_wav).name}")
    display(Audio(ref_wav))
    
    print("\n" + "="*60)
    print("Generated speech:")
    print("="*60)
    
    for i, text in enumerate(test_texts):
        out_path = f"/kaggle/working/ft_test_{i}.wav"
        tts.tts_to_file(
            text=text,
            speaker_wav=ref_wav,
            language="en",
            file_path=out_path,
        )
        print(f"\n{text}")
        display(Audio(out_path))

## Step 5: Export Model

Save the fine-tuned model as a Kaggle dataset so you can reuse it.

In [ ]:
import shutil

EXPORT_DIR = Path("/kaggle/working/export_model")
EXPORT_DIR.mkdir(exist_ok=True)

if best_model.exists():
    shutil.copy(best_model, EXPORT_DIR / "best_model.pth")
    shutil.copy(ft_config, EXPORT_DIR / "config.json")
    
    size_mb = best_model.stat().st_size / 1e6
    print(f"Model exported to {EXPORT_DIR} ({size_mb:.0f} MB)")
    print("\nTo reuse:")
    print("  1. Download from Kaggle output")
    print("  2. Upload as a new Kaggle dataset")
    print("  3. Attach to future notebooks")
else:
    print("No fine-tuned model to export. Run training first.")

## Key Takeaways

### What we learned:
1. **Data prep**: Clean audio + transcripts, 2-12s clips, 22050 Hz
2. **Fine-tuning**: Only the GPT-2 decoder is trained (adapter approach)
3. **Hyperparameters**: LR=5e-6, batch=2-4, epochs=5-20
4. **Evaluation**: Compare with base model using same reference audio

### Tips for best results:
- More data = better (30 min ideal, 10 min minimum)
- Clean audio is crucial — no background noise
- Train longer (20+ epochs) for production quality
- Use multiple reference clips for more consistent output

### Next steps:
- Try with your own voice recordings
- Experiment with different speakers from LibriTTS
- Try cross-lingual fine-tuning (train English, test Vietnamese)